# FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa — Exploration with `mlcroissant`

This notebook demonstrates how to load, overview, and process the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library. You will learn how to examine the dataset's metadata, investigate available record sets using their `@id`s, extract and manipulate tabular data, and perform simple exploratory data analysis (EDA).

### Dataset Source
The dataset Croissant schema is available at:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Install mlcroissant if needed
!pip install -U mlcroissant

## 1. Data Loading

In this section, we load the dataset's metadata and records using the `mlcroissant` library.

- The Croissant schema is accessed via the provided URL.
- We print the main metadata fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset via Croissant
dataset = mlc.Dataset(croissant_url)
# Display dataset name and description
print("Dataset Name:", dataset.metadata.name)
print("Description:\n", dataset.metadata.description)
print(f"Published: {getattr(dataset.metadata, 'datePublished', 'N/A')}")
print(f"Version: {getattr(dataset.metadata, 'version', 'N/A')}")
print(f"License: {getattr(dataset.metadata, 'license', 'N/A')}")

## 2. Data Overview

Let's review the available record sets, their `@id`s, fields, and columns. Referencing all entities by their `@id` ensures reproducibility and clarity.

- We will list all record sets available in the dataset.

In [ ]:
# List available RecordSets and their fields by their `@id`s
print("Record sets in dataset:")
record_sets = []
for rs in dataset.record_sets:
    print(f"- Record set name: {getattr(rs, 'name', '')}")
    print(f"  @id: {rs.id}")
    fields = getattr(rs, 'fields', [])
    if fields:
        print("  Fields and their @id:")
        for fld in fields:
            print(f"    - {getattr(fld, 'name', '')} (@id: {fld.id})")
    print()
    record_sets.append(rs.id)
if not record_sets:
    print("No record sets found in metadata.")

Below is a sample of records from the main record set for inspection.

*(Note: In this dataset, the main survey table is typically named with an `@id` like `cr:RecordSet` or similar. Adjust below for the actual output above.)*

In [ ]:
# Select primary record set by @id (update as needed)
primary_record_set = record_sets[0] if record_sets else None

if primary_record_set:
    print(f"Sample records from record set '{primary_record_set}':")
    for ix, record in enumerate(dataset.records(record_set=primary_record_set)):
        print(record)
        if ix >= 2:
            break
else:
    print("No record set found. Check dataset or Croissant schema.")

## 3. Data Extraction

Now, let's load the data from each record set into pandas DataFrames for analysis.

We'll use the list of record set `@id`s produced above and demonstrate extracting the first table. All references use the explicit `@id` as required.

In [ ]:
# Extract all record sets into DataFrames by their @id
dataframes = {}
for rs_id in record_sets:
    print(f"Loading records for record set: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(df.head(2))
    except Exception as e:
        print(f"  Could not load records: {e}")

# For subsequent analysis, select the main table
main_rs_id = primary_record_set
df = dataframes[main_rs_id]
print(f"Columns in record set '{main_rs_id}':")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)

Let's demonstrate several EDA techniques on a numeric field from the dataset. The column to use will be referenced strictly by its `@id` (which, for mental health survey datasets, is often like `gad_7_score`, `phq_9_score`, or similar). Adjust the code to reference whatever fields appear in your DataFrame columns above. Below is a generic example for a GAD-7 score-like field.

In [ ]:
# List all columns for reference (@id)
print(f"Columns in '{main_rs_id}':", df.columns.tolist())

# Choose a numeric field — set the correct @id!
# For this mental health dataset, common fields are: gad_7_score, phq_9_score, psq_score
numeric_field_id = None
for col in df.columns:
    if "gad" in col.lower() and "score" in col.lower():
        numeric_field_id = col
        break
if not numeric_field_id and len(df.select_dtypes('number').columns):
    numeric_field_id = df.select_dtypes('number').columns[0]

if numeric_field_id is None:
    raise ValueError("No obvious numeric mental health score field found.")

print(f"Using numeric field: {numeric_field_id}")

# Remove outliers above the 99th percentile
threshold = df[numeric_field_id].quantile(0.99)
filtered_df = df[df[numeric_field_id] <= threshold].copy()
print(f"Filtered records where {numeric_field_id} ≤ {threshold:.2f} (99th percentile):\n", filtered_df.head())

# Normalize the numeric field (z-score normalization)
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a demographic field (e.g., 'level_of_education' or similar @id)
group_field = None
options = [col for col in df.columns if any(x in col.lower() for x in ["education", "gender", "village", "marital"])]
if options:
    group_field = options[0]
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"Mean {numeric_field_id} by {group_field}:")
    print(grouped_df.head())

## 5. Visualization

We can now visualize field distributions and group-level comparisons. All columns are referenced by their `@id`, as above.

Example: Histogram and box plot of normalized GAD-7 scores, and comparison by education group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of normalized GAD-7/PHQ-9/PSQ scores
plt.figure(figsize=(7,4))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], bins=20, kde=True)
plt.title(f"Distribution of normalized {numeric_field_id}")
plt.xlabel(f"{numeric_field_id} (normalized)")
plt.ylabel("Count")
plt.show()

# Boxplot by group (e.g., level_of_education) if available
if group_field:
    plt.figure(figsize=(8,4))
    sns.boxplot(data=filtered_df, x=group_field, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.xticks(rotation=30, ha='right')
    plt.show()

## 6. Conclusion

- We loaded and explored the FAIR² mental health survey dataset using the `mlcroissant` library, referencing all record sets and fields by their `@id`s.
- After examining the metadata and available tables, we extracted records with their original, canonical field names.
- Exploratory analysis showed how to filter, normalize and visualize mental health assessment scores, and to investigate group-level differences by key demographics.

This approach can be adapted to any dataset described via a Croissant schema; simply adjust the `@id` references for your context.

For more details: [mlcroissant documentation](https://mlcommons.github.io/croissant/)
